# GLM protocol comparison: inference vs. application

First-level GLMs for `sub-01` (ses-03, task `arc`), one per combination of
**acquisition sequence** (`2p0mm64ch`, `2p5mm64ch`, `3p0mm64ch`) and
**reconstruction** (raw vs. NORDIC-denoised) &mdash; 6 GLMs in total.

Each GLM computes four contrasts:
- `inference_gt_application`: `inference - application`
- `task_gt_baseline`: `(inference + application) / 2` &mdash; either trial type vs. implicit baseline
- `inference_gt_baseline`: `inference` vs. implicit baseline
- `application_gt_baseline`: `application` vs. implicit baseline

(`application - inference` is dropped as it's just the sign-flip of
`inference_gt_application`.) Maps are thresholded at **p < .001, uncorrected,
one-tailed** (`height_control="fpr"`, `two_sided=False` in
`nilearn.glm.threshold_stats_img`) &mdash; no multiple-comparisons correction,
as requested.

Runs to include are discovered dynamically from which `_events.tsv` files
actually exist (a run with no matched behavioural log, e.g. an aborted scan,
has no events file and is silently skipped). This stands in for the scan
manifest CSV described in `CLAUDE.md`, which doesn't exist yet.

Preprocessed BOLD is fMRIPrep's `space-MNI152NLin2009cAsym_res-2` output.
Confounds are the standard 24 motion parameters, raw CSF/WM signal, and
fMRIPrep's cosine high-pass regressors, selected explicitly by column name
from each run's `desc-confounds_timeseries.tsv` (resolved via `pybids`, not
`nilearn.interfaces.fmriprep.load_confounds`'s own file lookup: that lookup
globs for a confounds file by stripping `space`/`desc`/`res` etc. from the
BOLD filename, and since it also strips `rec-nordic`, it matches *both* the
raw and NORDIC confounds files for the same run and raises. No scrubbing /
volume censoring is applied). Because the cosine regressors already
high-pass filter the data, the model's own drift model is turned off
(`drift_model=None`) to avoid duplicating that filtering.

An 8 mm FWHM Gaussian smoothing kernel is applied before model fitting &mdash;
fairly strong, deliberately so given how little data each protocol has here.

Maps are viewed on the standard MNI152 anatomical template (`view_img`'s
default `bg_img`), matching the `MNI152NLin2009cAsym` analysis space.

In [1]:
import os
from pathlib import Path

import pandas as pd
from bids import BIDSLayout
from dotenv import load_dotenv
from nilearn.glm import threshold_stats_img
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import intersect_masks
from nilearn.plotting import view_img

load_dotenv()
bids_root = Path(os.environ["BIDS_ROOT_DIR"])

layout = BIDSLayout(bids_root, validate=False, derivatives=True)

Matplotlib is building the font cache; this may take a moment.


In [2]:
SUBJECT = "01"
SESSION = "03"
TASK = "arc"
SPACE = "MNI152NLin2009cAsym"

ACQUISITIONS = ["2p0mm64ch", "2p5mm64ch", "3p0mm64ch"]
RECONSTRUCTIONS = [None, "nordic"]  # None = raw (no NORDIC)

CONTRASTS = {
    "inference_gt_application": "inference - application",
    "task_gt_baseline": "(inference + application) / 2",
    "inference_gt_baseline": "inference",
    "application_gt_baseline": "application",
}
ALPHA = 0.001
SMOOTHING_FWHM_MM = 8.0

OUT_DIR = bids_root / "derivatives" / "glm_protocol_comparison"
HTML_DIR = OUT_DIR / "htmls"
HTML_DIR.mkdir(parents=True, exist_ok=True)

## Helpers

`reconstruction` and `run` are filtered manually (rather than passed as
`layout.get(...)` keyword filters) so that `None` unambiguously means *the
entity is absent from the filename*, not *unfiltered*.

In [3]:
MOTION_PARAMS = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
MOTION_COLUMNS = (
    MOTION_PARAMS
    + [f"{p}_derivative1" for p in MOTION_PARAMS]
    + [f"{p}_power2" for p in MOTION_PARAMS]
    + [f"{p}_derivative1_power2" for p in MOTION_PARAMS]
)
WM_CSF_COLUMNS = ["white_matter", "csf"]


def get_runs(acq, rec):
    events = layout.get(
        subject=SUBJECT, session=SESSION, task=TASK, acquisition=acq,
        suffix="events", extension=".tsv",
    )
    events = [f for f in events if f.entities.get("reconstruction") == rec]
    return sorted({f.entities.get("run") for f in events}, key=lambda r: (r is not None, r or ""))


def get_single(acq, rec, run, **filters):
    kwargs = dict(subject=SUBJECT, session=SESSION, task=TASK, acquisition=acq, **filters)
    if run is not None:
        kwargs["run"] = run
    matches = [
        f for f in layout.get(**kwargs)
        if f.entities.get("reconstruction") == rec and f.entities.get("run") == run
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"expected exactly one match for acq={acq} rec={rec} run={run} "
            f"filters={filters}, got {len(matches)}"
        )
    return matches[0]


def read_confounds(confounds_tsv_path):
    df = pd.read_csv(confounds_tsv_path, sep="\t")
    cosine_columns = [c for c in df.columns if c.startswith("cosine")]
    columns = MOTION_COLUMNS + WM_CSF_COLUMNS + cosine_columns
    return df[columns].fillna(0)  # derivative columns are NaN on the first row


def fit_glm(acq, rec):
    runs = get_runs(acq, rec)
    events_list, bold_list, confounds_list, masks, trs = [], [], [], [], []

    for run in runs:
        ev_f = get_single(acq, rec, run, suffix="events", extension=".tsv")
        events_list.append(pd.read_csv(ev_f.path, sep="\t")[["onset", "duration", "trial_type"]])

        bold_f = get_single(
            acq, rec, run, space=SPACE, desc="preproc", suffix="bold", extension=".nii.gz",
        )
        bold_list.append(bold_f.path)
        trs.append(bold_f.get_metadata()["RepetitionTime"])

        mask_f = get_single(
            acq, rec, run, space=SPACE, desc="brain", suffix="mask", extension=".nii.gz",
        )
        masks.append(mask_f.path)

        confounds_f = get_single(acq, rec, run, desc="confounds", suffix="timeseries", extension=".tsv")
        confounds_list.append(read_confounds(confounds_f.path))

    assert len(set(trs)) == 1, f"inconsistent TR across runs for {acq}/{rec}: {trs}"
    mask_img = masks[0] if len(masks) == 1 else intersect_masks(masks, threshold=1)

    model = FirstLevelModel(
        t_r=trs[0],
        hrf_model="glover",
        drift_model=None,  # fMRIPrep's cosine confound regressors already high-pass filter
        mask_img=mask_img,
        smoothing_fwhm=SMOOTHING_FWHM_MM,
        standardize=False,
        minimize_memory=True,
    )
    model = model.fit(bold_list, events=events_list, confounds=confounds_list)
    return model, runs

## Fit all 6 GLMs and compute all four contrasts

`nilearn` will warn that "the same contrast will be used for all N runs" for
the `2p0mm64ch` combinations (2 runs) &mdash; expected, since both runs of a
combination share the same design.

In [4]:
z_maps = {}
summary_rows = []

for acq in ACQUISITIONS:
    for rec in RECONSTRUCTIONS:
        label = f"{acq} / {'NORDIC' if rec else 'no NORDIC'}"
        print(f"Fitting {label} ...")
        model, runs = fit_glm(acq, rec)

        for contrast_key, contrast_def in CONTRASTS.items():
            z_map = model.compute_contrast(contrast_def, output_type="z_score")
            _, z_thresh = threshold_stats_img(z_map, alpha=ALPHA, height_control="fpr", two_sided=False)
            n_suprathreshold = int((z_map.get_fdata() > z_thresh).sum())

            z_maps[(acq, rec, contrast_key)] = (z_map, z_thresh)

            summary_rows.append({
                "acquisition": acq,
                "reconstruction": "nordic" if rec else "raw",
                "contrast": contrast_key,
                "n_runs": len(runs),
                "z_threshold": round(z_thresh, 3),
                "n_suprathreshold_voxels": n_suprathreshold,
            })

summary = pd.DataFrame(summary_rows)
summary

Fitting 2p0mm64ch / no NORDIC ...


/tmp/ipykernel_241784/3795739624.py:77: RuntimeWarning: [MultiNiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  model = model.fit(bold_list, events=events_list, confounds=confounds_list)


/tmp/ipykernel_241784/681772255.py:11: RuntimeWarning: The same contrast will be used for all 2 runs. If the design matrices are not the same for all runs, (for example with different column names or column order across runs) you should pass contrast as an expression using the name of the conditions as they appear in the design matrices.
  z_map = model.compute_contrast(contrast_def, output_type="z_score")


/tmp/ipykernel_241784/681772255.py:11: RuntimeWarning: The same contrast will be used for all 2 runs. If the design matrices are not the same for all runs, (for example with different column names or column order across runs) you should pass contrast as an expression using the name of the conditions as they appear in the design matrices.
  z_map = model.compute_contrast(contrast_def, output_type="z_score")


/tmp/ipykernel_241784/681772255.py:11: RuntimeWarning: The same contrast will be used for all 2 runs. If the design matrices are not the same for all runs, (for example with different column names or column order across runs) you should pass contrast as an expression using the name of the conditions as they appear in the design matrices.
  z_map = model.compute_contrast(contrast_def, output_type="z_score")


...

Fitting 3p0mm64ch / no NORDIC ...


/tmp/ipykernel_241784/3795739624.py:77: RuntimeWarning: [MultiNiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  model = model.fit(bold_list, events=events_list, confounds=confounds_list)


Fitting 3p0mm64ch / NORDIC ...


/tmp/ipykernel_241784/3795739624.py:77: RuntimeWarning: [MultiNiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  model = model.fit(bold_list, events=events_list, confounds=confounds_list)


,acquisition,reconstruction,contrast,n_runs,z_threshold,n_suprathreshold_voxels
0,2p0mm64ch,raw,inference_gt_application,2,3.09,3023
1,2p0mm64ch,raw,task_gt_baseline,2,3.09,878
2,2p0mm64ch,raw,inference_gt_baseline,2,3.09,2502
3,2p0mm64ch,raw,application_gt_baseline,2,3.09,51
4,2p0mm64ch,nordic,inference_gt_application,2,3.09,2158
5,2p0mm64ch,nordic,task_gt_baseline,2,3.09,1829
6,2p0mm64ch,nordic,inference_gt_baseline,2,3.09,2739
7,2p0mm64ch,nordic,application_gt_baseline,2,3.09,218
8,2p5mm64ch,raw,inference_gt_application,1,3.09,754
9,2p5mm64ch,raw,task_gt_baseline,1,3.09,91


## Interactive HTML report

Each thresholded map gets its own standalone interactive viewer
(`nilearn.plotting.view_img`, saved via `save_as_html`). `index.html` wraps
all 24 (6 GLMs &times; 4 contrasts) in a single page with three independent
dropdowns &mdash; acquisition, reconstruction, contrast &mdash; that combine
to pick the matching viewer. The viewer is scaled (not scrolled) to fit
whatever space is available in the browser window, so nothing is ever
occluded or requires scrolling.

If your browser refuses to load the iframe from a `file://` URL, serve the
folder instead, e.g. `python -m http.server` from `glm_protocol_comparison/`
and open `http://localhost:8000/index.html`.

In [5]:
html_names = []

for (acq, rec, contrast_key), (z_map, z_thresh) in z_maps.items():
    contrast_def = CONTRASTS[contrast_key]
    label = f"{acq} / {'NORDIC' if rec else 'no NORDIC'}"

    view = view_img(
        z_map, threshold=z_thresh,
        title=f"{label}: {contrast_def} (z > {z_thresh:.2f}, p < {ALPHA} unc.)",
    )
    html_name = f"{acq}_{'nordic' if rec else 'raw'}_{contrast_key}.html"
    view.save_as_html(HTML_DIR / html_name)
    html_names.append(html_name)

summary["html"] = html_names
summary

/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


...

/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


/gpfs01/bartels/user/mbannert/miniconda3/envs/ARC-fMRI/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


,acquisition,reconstruction,contrast,n_runs,z_threshold,n_suprathreshold_voxels,html
0,2p0mm64ch,raw,inference_gt_application,2,3.09,3023,2p0mm64ch_raw_inference_gt_application.html
1,2p0mm64ch,raw,task_gt_baseline,2,3.09,878,2p0mm64ch_raw_task_gt_baseline.html
2,2p0mm64ch,raw,inference_gt_baseline,2,3.09,2502,2p0mm64ch_raw_inference_gt_baseline.html
3,2p0mm64ch,raw,application_gt_baseline,2,3.09,51,2p0mm64ch_raw_application_gt_baseline.html
4,2p0mm64ch,nordic,inference_gt_application,2,3.09,2158,2p0mm64ch_nordic_inference_gt_application.html
5,2p0mm64ch,nordic,task_gt_baseline,2,3.09,1829,2p0mm64ch_nordic_task_gt_baseline.html
6,2p0mm64ch,nordic,inference_gt_baseline,2,3.09,2739,2p0mm64ch_nordic_inference_gt_baseline.html
7,2p0mm64ch,nordic,application_gt_baseline,2,3.09,218,2p0mm64ch_nordic_application_gt_baseline.html
8,2p5mm64ch,raw,inference_gt_application,1,3.09,754,2p5mm64ch_raw_inference_gt_application.html
9,2p5mm64ch,raw,task_gt_baseline,1,3.09,91,2p5mm64ch_raw_task_gt_baseline.html


In [6]:
def build_index_html(out_path: Path):
    acq_options = "\n".join(f'<option value="{a}">{a}</option>' for a in ACQUISITIONS)
    rec_options = "\n".join(
        f'<option value="{r}">{"NORDIC" if r == "nordic" else "raw"}</option>' for r in ["raw", "nordic"]
    )
    contrast_options = "\n".join(f'<option value="{c}">{c}</option>' for c in CONTRASTS)

    default_src = f"htmls/{ACQUISITIONS[0]}_raw_{next(iter(CONTRASTS))}.html"

    # nilearn's brainsprite viewer sizes its canvas from its container's clientWidth alone and
    # derives the height from a *fixed* ratio set by the MNI grid's slice counts (~2.45:1 here,
    # constant across all 24 files since they share the same res-2 MNI152 grid) -- it has no
    # notion of the available height, so on a wide-but-short window the canvas overflows
    # vertically and the iframe grows a scrollbar. Rendering the iframe at a fixed native
    # resolution matching that ratio and then CSS-scaling the whole iframe down (or up) to fit
    # whatever space is available, recomputed on every resize, sidesteps that entirely: the
    # brain images never get taller than the window, and nothing ever needs to scroll.
    html = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>ARC-fMRI protocol comparison: inference vs. application</title>
<style>
  body {{ font-family: sans-serif; margin: 0; display: flex; flex-direction: column; height: 100vh;
          overflow: hidden; }}
  header {{ padding: 0.75rem 1rem; border-bottom: 1px solid #ccc; display: flex; gap: 1.5rem;
            align-items: center; flex-wrap: wrap; }}
  label {{ display: flex; gap: 0.4rem; align-items: center; }}
  select {{ font-size: 1rem; padding: 0.25rem; }}
  #viewerWrap {{ flex: 1; position: relative; overflow: hidden; }}
  #viewer {{ position: absolute; top: 0; left: 0; width: 1600px; height: 654px; border: none;
             transform-origin: top left; }}
</style>
</head>
<body>
<header>
  <label>Acquisition <select id="acq">{acq_options}</select></label>
  <label>Reconstruction <select id="rec">{rec_options}</select></label>
  <label>Contrast <select id="contrast">{contrast_options}</select></label>
</header>
<div id="viewerWrap">
  <iframe id="viewer" src="{default_src}"></iframe>
</div>
<script>
  var VIEWER_NATIVE_WIDTH = 1600;
  var VIEWER_NATIVE_HEIGHT = 654;

  function fitViewer() {{
    var wrap = document.getElementById('viewerWrap');
    var iframe = document.getElementById('viewer');
    var scale = Math.min(wrap.clientWidth / VIEWER_NATIVE_WIDTH, wrap.clientHeight / VIEWER_NATIVE_HEIGHT);
    var scaledWidth = VIEWER_NATIVE_WIDTH * scale;
    var scaledHeight = VIEWER_NATIVE_HEIGHT * scale;
    iframe.style.transform = 'scale(' + scale + ')';
    iframe.style.left = ((wrap.clientWidth - scaledWidth) / 2) + 'px';
    iframe.style.top = ((wrap.clientHeight - scaledHeight) / 2) + 'px';
  }}

  function updateViewer() {{
    var acq = document.getElementById('acq').value;
    var rec = document.getElementById('rec').value;
    var contrast = document.getElementById('contrast').value;
    document.getElementById('viewer').src = 'htmls/' + acq + '_' + rec + '_' + contrast + '.html';
    fitViewer();
  }}
  document.getElementById('acq').addEventListener('change', updateViewer);
  document.getElementById('rec').addEventListener('change', updateViewer);
  document.getElementById('contrast').addEventListener('change', updateViewer);
  window.addEventListener('resize', fitViewer);
  fitViewer();
</script>
</body>
</html>
"""
    with open(out_path, "w") as f:
        f.write(html)


index_path = OUT_DIR / "index.html"
build_index_html(index_path)
print(f"Interactive report: {index_path}")

Interactive report: /gpfs01/bartels/group/mbannert/projects/ARC-fMRI/bids/derivatives/glm_protocol_comparison/index.html
